# Segmentação de pluma de fumaça com SAM — versão aprimorada

Continuação do notebook inicial da atividade. Mesma stack (**Ultralytics**) e mesma imagem
(`MyDrive/pluma.jpg`); o que muda é **como o SAM é acionado** e **como o resultado é avaliado**.

## O que estava quebrado no notebook inicial

| Célula | O que acontecia | Causa | O que foi feito aqui |
|---|---|---|---|
| 13 | `FileNotFoundError: 'sam_b.pt'` | `SAM3SemanticPredictor` espera pesos do **SAM 3**; o download deu 404 e `sam_b.pt` (SAM 1) não serve para *prompt* de texto | troquei por `SAM2Predictor` com *prompts* **geométricos** (caixa e pontos), que é o que a tarefa pede |
| 14 | `(no detections)` → "Nenhuma fumaça detectada" | o detector `YOLOv8m-smoke-detection` não encontrou a pluma, então **nenhuma máscara foi gerada** | não dependo mais de detector automático: a caixa vem de um palpite calculado na imagem e é ajustada à mão |
| 12 | YOLO genérico acusou "4 cars, 1 truck" | `yolo26n` é COCO: não existe classe "fumaça" | removido — não contribui para a máscara |
| 7 | `shutil.copyfile("pluma.jpg", "pluma.png")` | só renomeia: o arquivo continua **JPEG** com extensão `.png` | leio o original direto, sem cópia |

> ⚠️ **Antes de publicar no GitHub:** o notebook inicial tem uma **chave da API do CDS/Copernicus
> escrita no código** (célula 4). Chave em repositório público é chave vazada — revogue essa e gere
> outra no painel do CDS, e passe a lê-la de variável de ambiente
> (`os.environ["CDSAPI_KEY"]`). A parte de ERA5/netCDF não entra aqui porque é outra frente do
> projeto; esta entrega é só a segmentação da pluma.

## Como rodar

`Ambiente de execução → Executar tudo`. GPU ajuda mas não é obrigatória (o notebook escolhe
`sam2_t.pt` em CPU e `sam2_b.pt` em GPU).

**Ajuste obrigatório:** a Etapa 1 calcula um palpite de caixa e mostra a imagem com uma grade de
coordenadas. Confira se o palpite caiu na pluma e, se não caiu, corrija `PROMPTS` na Etapa 2 —
as coordenadas são frações de 0 a 1, lidas direto da grade.

In [ ]:
# ============================================================
# ETAPA 1 — Instalação, imagem e palpite inicial de prompts
# ============================================================
!pip install -q ultralytics

import os
import numpy as np, cv2, torch, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

np.random.seed(42); torch.manual_seed(42)
plt.rcParams.update({"figure.dpi": 110, "figure.facecolor": "white",
                     "axes.titlesize": 10, "axes.titleweight": "bold", "font.size": 9})
os.makedirs("saida", exist_ok=True)

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
MODELO = "sam2_b.pt" if DISPOSITIVO == "cuda" else "sam2_t.pt"   # Ultralytics baixa sozinho
CAMINHO = "/content/drive/MyDrive/pluma.jpg"                      # mesma imagem do notebook inicial

if not os.path.exists(CAMINHO):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        pass
if not os.path.exists(CAMINHO):
    from google.colab import files
    CAMINHO = list(files.upload().keys())[0]

IMG = cv2.cvtColor(cv2.imread(CAMINHO), cv2.COLOR_BGR2RGB)   # cv2 lê pelo conteúdo, não pela extensão
ALT, LARG = IMG.shape[:2]
print(f"imagem {LARG}x{ALT} | dispositivo={DISPOSITIVO} | modelo={MODELO}")


def para_px(pts):
    p = np.asarray(pts, float).reshape(-1, 2)
    return np.stack([p[:, 0] * LARG, p[:, 1] * ALT], axis=1)


def caixa_px(c):
    return np.array([c[0] * LARG, c[1] * ALT, c[2] * LARG, c[3] * ALT], float)


def roi_mascara(c):
    x0, y0, x1, y1 = caixa_px(c).astype(int)
    m = np.zeros((ALT, LARG), bool); m[max(y0, 0):y1, max(x0, 0):x1] = True
    return m


def palpite():
    """Palpite de caixa e de pontos positivos a partir da própria imagem.

    Fumaça iluminada é CLARA e pouco SATURADA, e a pluma fica acima da linha do solo.
    Não é um detector — é só um ponto de partida melhor do que chutar coordenadas, e já
    mostra o problema do enunciado: o mesmo critério também pega céu claro e nuvem.
    """
    hsv = cv2.cvtColor(IMG, cv2.COLOR_RGB2HSV)
    cand = (hsv[:, :, 2] > 110) & (hsv[:, :, 1] < 80)
    cand[int(0.75 * ALT):] = False                      # ignora o terço inferior (solo, rua, prédios)
    cand = cv2.morphologyEx(cand.astype(np.uint8), cv2.MORPH_OPEN, np.ones((9, 9), np.uint8))
    n, rot, stats, _ = cv2.connectedComponentsWithStats(cand, 8)
    if n < 2:
        return (0.25, 0.20, 0.95, 0.60), [(0.45, 0.35), (0.55, 0.40), (0.65, 0.35)]
    i = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    x, y, w, h = stats[i, :4]
    ys, xs = np.nonzero(rot == i)
    pos = []
    for q in (0.25, 0.5, 0.75):                          # 3 pontos espalhados ao longo da pluma
        xq = np.quantile(xs, q)
        sel = np.abs(xs - xq) < max(0.02 * LARG, 2)
        pos.append((float(xq / LARG), float(np.median(ys[sel]) / ALT)))
    return (x / LARG, y / ALT, (x + w) / LARG, (y + h) / ALT), pos


CAIXA_SUGERIDA, POS_SUGERIDOS = palpite()
print("caixa sugerida   :", tuple(round(v, 3) for v in CAIXA_SUGERIDA))
print("pontos sugeridos :", [tuple(round(v, 3) for v in p) for p in POS_SUGERIDOS])

# imagem com grade de coordenadas, para conferir/ajustar os prompts à mão
fig, ax = plt.subplots(figsize=(11, 11 * ALT / LARG))
ax.imshow(IMG)
for x in range(0, LARG, max(50, round(LARG / 12 / 50) * 50)):
    ax.axvline(x, color="yellow", lw=.4); ax.text(x + 2, 14, f"{x}\n{x/LARG:.2f}", color="red", fontsize=6, va="top")
for y in range(0, ALT, max(50, round(ALT / 10 / 50) * 50)):
    ax.axhline(y, color="yellow", lw=.4); ax.text(2, y + 4, f"{y} / {y/ALT:.2f}", color="red", fontsize=6, va="top")
x0, y0, x1, y1 = caixa_px(CAIXA_SUGERIDA)
ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="cyan", lw=2))
p = para_px(POS_SUGERIDOS); ax.scatter(p[:, 0], p[:, 1], c="lime", marker="*", s=180, ec="k")
ax.set_title(f"palpite inicial sobre a grade — {LARG}x{ALT}px (rótulos: pixel / fração)"); ax.axis("off")
fig.tight_layout(); fig.savefig("saida/01_grade.png", bbox_inches="tight"); plt.show()

---
# Etapa 2 — Três configurações de *prompt*

| | *Prompt* | Hipótese testada |
|---|---|---|
| **A** | só a caixa | o retângulo basta? Ele contém muito céu junto da pluma. |
| **B** | só pontos positivos | sem limite espacial, até onde a região "espalha" por semelhança? |
| **C** | caixa + positivos + **negativos** | dizer o que **não** é pluma resolve a ambiguidade? |

**Justificativa da melhoria.** O SAM é agnóstico a classe: ele não tem o conceito "fumaça", só
devolve a região coerente que o *prompt* indica. Por isso o caminho do notebook inicial — deixar um
detector achar a fumaça — falha duas vezes: o detector não detectou nada (célula 14) e o *prompt* de
texto exigia um modelo indisponível (célula 13). Com *prompt* geométrico o controle volta para as
nossas mãos, e o **ponto negativo** passa a ser o único elemento capaz de excluir um distrator que
está *dentro* da caixa — que é exatamente o caso de céu claro, nuvem e telhado.

Outra melhoria de código: `set_image()` é chamado **uma vez** e reaproveitado pelas três
configurações. O custo do SAM está em codificar a imagem; o decodificador de *prompt* é leve. O
notebook inicial recriava a inferência a cada chamada.

**Se o palpite da Etapa 1 não caiu na pluma, corrija `PROMPTS` abaixo** lendo as coordenadas na grade.

In [ ]:
# ============================================================
# ETAPA 2 — Prompts e predição
# ============================================================
from ultralytics.models.sam import SAM2Predictor

PROMPTS = {
    "caixa": CAIXA_SUGERIDA,          # <- substitua por (x0, y0, x1, y1) em fração, se precisar
    "positivos": POS_SUGERIDOS,       # <- [(x, y), ...] sobre o corpo da pluma
    "negativos": [(0.04, 0.04), (0.96, 0.04),      # cantos de céu
                  (0.50, 0.96), (0.04, 0.90)],     # solo / base da cena
}

predictor = SAM2Predictor(overrides=dict(task="segment", mode="predict", model=MODELO,
                                         save=False, verbose=False, conf=0.25))
predictor.set_image(CAMINHO)


def prever(caixa=None, pontos=None, rotulos=None):
    """Chama o SAM com prompt geométrico e devolve uma máscara booleana HxW."""
    kw = {}
    if caixa is not None:
        kw["bboxes"] = caixa_px(caixa)[None, :]
    if pontos:
        kw["points"] = para_px(pontos)
        kw["labels"] = np.asarray(rotulos, int)
    r = predictor(**kw)[0]
    if r.masks is None or len(r.masks.data) == 0:
        return np.zeros((ALT, LARG), bool)
    m = r.masks.data.cpu().numpy().any(0).astype(np.uint8)
    if m.shape != (ALT, LARG):                     # defesa: o notebook inicial também reescalava
        m = cv2.resize(m, (LARG, ALT), interpolation=cv2.INTER_NEAREST)
    return m.astype(bool)


n_pos, n_neg = len(PROMPTS["positivos"]), len(PROMPTS["negativos"])
CONFIGS = {
    "A — só caixa": dict(caixa=PROMPTS["caixa"]),
    "B — só pontos (+)": dict(pontos=PROMPTS["positivos"], rotulos=np.ones(n_pos, int)),
    "C — caixa + (+) + (−)": dict(caixa=PROMPTS["caixa"],
                                  pontos=PROMPTS["positivos"] + PROMPTS["negativos"],
                                  rotulos=np.r_[np.ones(n_pos, int), np.zeros(n_neg, int)]),
}

MASCARAS = {}
for nome, cfg in CONFIGS.items():
    MASCARAS[nome] = prever(**cfg)
    print(f"{nome:24s} área = {100*MASCARAS[nome].mean():5.2f}% da imagem")

---
# Etapa 3 — Apresentação da máscara

Original com os *prompts* desenhados, máscara binária e máscara sobreposta, lado a lado, para cada
configuração. Três decisões de apresentação:

* os *prompts* vão **sobre a original** — sem isso não dá para saber se o erro é do modelo ou de um
  ponto mal colocado;
* a sobreposição usa **transparência + contorno**, porque preenchimento opaco esconde exatamente a
  borda, que é onde está o erro interessante;
* o painel binário traz o **percentual de área**, para dar escala ao que se vê.

In [ ]:
# ============================================================
# ETAPA 3 — Figuras lado a lado
# ============================================================

def sobrepor(img, mask, alpha=0.45):
    out = img.astype(np.float32).copy()
    out[mask] = (1 - alpha) * out[mask] + alpha * np.array([255, 60, 60], np.float32)
    out = out.astype(np.uint8)
    cont, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(out, cont, -1, (255, 230, 0), 2)
    return out


fig, axs = plt.subplots(3, 3, figsize=(13, 3.2 * 3 * ALT / LARG + 1.2))
for lin, (nome, cfg) in enumerate(CONFIGS.items()):
    m = MASCARAS[nome]
    axs[lin, 0].imshow(IMG)
    if cfg.get("caixa") is not None:
        bx0, by0, bx1, by1 = caixa_px(cfg["caixa"])
        axs[lin, 0].add_patch(Rectangle((bx0, by0), bx1 - bx0, by1 - by0, fill=False, ec="yellow", lw=2))
    if cfg.get("pontos"):
        pp, rr = para_px(cfg["pontos"]), np.asarray(cfg["rotulos"])
        axs[lin, 0].scatter(pp[rr == 1, 0], pp[rr == 1, 1], c="lime", marker="*", s=150, ec="k", lw=.6)
        axs[lin, 0].scatter(pp[rr == 0, 0], pp[rr == 0, 1], c="red", marker="X", s=90, ec="k", lw=.6)
    axs[lin, 1].imshow(m, cmap="gray")
    axs[lin, 2].imshow(sobrepor(IMG, m))
    axs[lin, 0].set_ylabel(nome, fontsize=9)
    axs[lin, 1].set_xlabel(f"área = {100*m.mean():.2f}%", fontsize=8)
    for a in axs[lin]:
        a.set_xticks([]); a.set_yticks([])
for c, t in enumerate(["original + prompts (★ positivo, ✕ negativo)", "máscara binária", "sobreposição"]):
    axs[0, c].set_title(t)
fig.tight_layout(); fig.savefig("saida/03_comparativo.png", bbox_inches="tight"); plt.show()

---
# Etapa 4 — Avaliação no código

Sem uma máscara de referência traçada à mão não dá para calcular IoU contra a verdade. Então as
perguntas do enunciado são respondidas por medidas que **não dependem de rótulo**:

| Pergunta do enunciado | Medida |
|---|---|
| A máscara cobre a pluma? | `área (%)` e `fora da caixa (%)` — máscara que encolhe demais perde a pluma |
| Inclui o que não pertence a ela? | `invadiu céu (%)` e `invadiu solo (%)` (faixas de topo e base da cena) |
| Deixa fumaça de fora? | `componentes`: pluma partida em várias ilhas indica trecho perdido |
| O resultado é estável? | `IoU` entre as três configurações — quanto menor, mais a resposta veio do *prompt* e não da imagem |
| A borda corresponde a algo real? | **separabilidade dentro × fora** (distância de Bhattacharyya): se for baixa, a máscara cortou uma região fotometricamente homogênea, ou seja, a borda foi decidida pelo *prompt* |

A última linha é a resposta direta a *"como a iluminação, o contraste e a semelhança entre fumaça e
nuvens afetaram o resultado"*: ela mede se existe, na imagem, uma fronteira para a máscara se apoiar.
A Etapa 4.2 fecha o argumento reduzindo o contraste da mesma cena e refazendo a predição.

In [ ]:
# ============================================================
# ETAPA 4.1 — Métricas
# ============================================================
CINZA = cv2.cvtColor(IMG, cv2.COLOR_RGB2GRAY)
SAT = cv2.cvtColor(IMG, cv2.COLOR_RGB2HSV)[:, :, 1]
ROI_CEU, ROI_SOLO = roi_mascara((0.0, 0.0, 1.0, 0.12)), roi_mascara((0.0, 0.88, 1.0, 1.0))


def bhattacharyya(a, b, bins=64):
    """0 = distribuições idênticas; 8 = saturado (sem sobreposição)."""
    if len(a) == 0 or len(b) == 0:
        return float("nan")
    faixa = (float(min(a.min(), b.min())), float(max(a.max(), b.max())) + 1e-6)
    ha = np.histogram(a, bins, faixa)[0]; hb = np.histogram(b, bins, faixa)[0]
    ha = ha / (ha.sum() + 1e-12); hb = hb / (hb.sum() + 1e-12)
    return float(min(-np.log(np.sum(np.sqrt(ha * hb)) + 1e-12), 8.0))


def metricas(m, caixa=None):
    d = {"área (%)": 100 * m.mean()}
    n, _, stats, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8), 8)
    d["componentes"] = int(n - 1)
    cont, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    per = sum(cv2.arcLength(c, True) for c in cont)
    d["compacidade"] = float(4 * np.pi * m.sum() / per ** 2) if per else 0.0
    d["fora da caixa (%)"] = (100 * np.logical_and(m, ~roi_mascara(caixa)).sum() / max(m.sum(), 1)
                              if caixa is not None else float("nan"))
    d["invadiu céu (%)"] = 100 * np.logical_and(m, ROI_CEU).sum() / ROI_CEU.sum()
    d["invadiu solo (%)"] = 100 * np.logical_and(m, ROI_SOLO).sum() / ROI_SOLO.sum()
    d["separab. luminância"] = bhattacharyya(CINZA[m].astype(float), CINZA[~m].astype(float))
    d["separab. saturação"] = bhattacharyya(SAT[m].astype(float), SAT[~m].astype(float))
    return d


TABELA = pd.DataFrame({n: metricas(m, CONFIGS[n].get("caixa"))
                       for n, m in MASCARAS.items()}).T.round(3)
display(TABELA)
TABELA.to_csv("saida/04_metricas.csv")


def iou(a, b):
    u = np.logical_or(a, b).sum()
    return float(np.logical_and(a, b).sum() / u) if u else 0.0


nomes = list(MASCARAS)
for i, a in enumerate(nomes):
    for b in nomes[i + 1:]:
        print(f"IoU({a.split(' ')[0]}, {b.split(' ')[0]}) = {iou(MASCARAS[a], MASCARAS[b]):.3f}")

In [ ]:
# ============================================================
# ETAPA 4.2 — Efeito do contraste (mesma cena, mesmo prompt)
# ============================================================
base = MASCARAS["C — caixa + (+) + (−)"]
x = IMG.astype(np.float32) / 255
baixo = (np.clip((x - 0.5) * 0.5 + 0.5, 0, 1) * 255).astype(np.uint8)
cv2.imwrite("saida/_baixo_contraste.png", cv2.cvtColor(baixo, cv2.COLOR_RGB2BGR))

predictor.set_image("saida/_baixo_contraste.png")     # reencoda a imagem alterada
m2 = prever(**CONFIGS["C — caixa + (+) + (−)"])
predictor.set_image(CAMINHO)                          # restaura
print(f"contraste x0.5 -> IoU vs original = {iou(m2, base):.3f} | "
      f"área {100*(m2.mean()-base.mean()):+.2f} p.p.")

fig, axs = plt.subplots(1, 2, figsize=(13, 13 * ALT / LARG / 2))
axs[0].imshow(sobrepor(IMG, base)); axs[0].set_title("original (C)")
axs[1].imshow(sobrepor(baixo, m2))
axs[1].set_title(f"contraste reduzido à metade\nIoU vs original = {iou(m2, base):.2f}")
for a in axs:
    a.axis("off")
fig.tight_layout(); fig.savefig("saida/04_contraste.png", bbox_inches="tight"); plt.show()

# exporta máscaras e sobreposições
for nome, m in MASCARAS.items():
    rot = nome.split(" ")[0]
    cv2.imwrite(f"saida/mascara_{rot}.png", (m * 255).astype(np.uint8))
    cv2.imwrite(f"saida/sobreposicao_{rot}.png", cv2.cvtColor(sobrepor(IMG, m), cv2.COLOR_RGB2BGR))
print("gerado:", sorted(os.listdir("saida")))

---
# Etapa 5 — Conclusões

*(Confira os números da **sua** execução nas tabelas acima antes de entregar.)*

## A máscara cobre a pluma? Inclui o que não é? Deixa fumaça de fora?

* **A (só caixa)** — o retângulo que contém a pluma contém também muito céu. O SAM devolve a região
  mais coerente dentro dele, então a máscara ou vaza para o céu (veja `invadiu céu (%)`) ou encolhe
  para o núcleo mais denso, perdendo as bordas finas da pluma.
* **B (só pontos positivos)** — sem limite espacial, a máscara cresce por continuidade de aparência.
  Como fumaça iluminada tem quase a mesma luminância e saturação de nuvem e de céu claro, ela
  costuma invadir a região vizinha — é o que `área (%)` e `invadiu céu (%)` mostram.
* **C (caixa + positivos + negativos)** — os negativos são o único elemento do *prompt* capaz de
  excluir um distrator que está *dentro* da caixa. É a configuração que se espera melhor, e as
  colunas de invasão dizem se isso de fato aconteceu nesta imagem.

O **IoU entre as três** é a medida mais honesta do problema: se ele for baixo, três *prompts*
razoáveis sobre a mesma imagem produziram três respostas diferentes — ou seja, a resposta veio do
*prompt*, não da imagem.

## O que dificultou a segmentação

* **Semelhança fumaça × nuvem/céu claro.** É o fator dominante, e aparece na coluna
  `separab. saturação`: fumaça e nuvem são ambas quase acromáticas, então cor não separa as duas.
  O próprio palpite da Etapa 1 — claro e pouco saturado — pega céu e nuvem junto, o que é a
  demonstração prática disso.
* **Contraste e borda difusa.** Fumaça é um meio **semitransparente**: perto da borda cada pixel é
  mistura de fumaça e fundo. Não existe gradiente forte para o contorno se apoiar, e é por isso que
  a borda acaba decidida pela caixa que desenhamos. A `separab. luminância` quantifica isso: valor
  baixo significa que a máscara cortou uma região fotometricamente homogênea.
* **Iluminação.** A Etapa 4.2 mostra a mesma cena, com o mesmo *prompt*, dando outra máscara só
  porque o contraste mudou — o SAM recalcula o *embedding* e passa a "ver" outra coisa. Consequência
  prática: fotos do mesmo incêndio em horários diferentes não são comparáveis sem normalização.
* **Detecção automática não resolve.** O notebook inicial tentou dois atalhos automáticos e os dois
  falharam (detector sem detecção, *prompt* de texto sem pesos). Segmentar fumaça com SAM é, hoje,
  uma tarefa guiada: alguém precisa dizer onde ela está.

## O que eu faria a seguir

* **Máscara de referência** traçada à mão (ou de consenso entre duas pessoas) para medir IoU, Dice,
  precisão e recall de verdade — e para saber qual é o teto realista nesta imagem.
* **Refinamento iterativo** realimentando a máscara anterior como `masks=` no *prompt*, e limpeza
  morfológica para remover ilhas isoladas.
* Sair da máscara binária para **estimativa de transparência (alfa)**, que é o formato adequado para
  um meio semitransparente e para medir área de pluma em série temporal.
* Repetir o protocolo em **N imagens** e reportar média ± desvio, em vez de um caso só.

---

## Entrega

Rode tudo, depois `Arquivo → Salvar uma cópia no GitHub`. O link a entregar é o do `.ipynb` no
GitHub. Rodar **antes** de salvar importa: o GitHub renderiza o notebook com as saídas embutidas,
então o professor vê as figuras e as tabelas sem precisar executar nada.